# Лаб 1. Решающее дерево ID3

In [70]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score

from source.model import tree_growing, predict, prune, accuracy, count_leaves, tree_depth

## Загрузка и подготовка данных

In [79]:
!curl -L -o ./dataset/Heart_Attack.csv https://www.kaggle.com/api/v1/datasets/download/nvarisha/heart-attack-data-analysis

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100  3510  100  3510    0     0   6525      0 --:--:-- --:--:-- --:--:--  6525


In [80]:
df = pd.read_csv('dataset/Heart_Attack.csv', encoding='utf-8-sig')
print(df.shape)
df.head()

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xf0 in position 14: invalid continuation byte

In [72]:
cat_columns = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']
num_columns = ['age', 'trestbps', 'chol', 'thalach', 'oldpeak']
target_col = 'target'

feature_names = [c for c in df.columns if c != target_col]
cat_features = set([feature_names.index(c) for c in cat_columns])

Вносим пропуски ~10% в 4 столбца

In [73]:
np.random.seed(42)
for col in ['age', 'chol', 'thal', 'ca']:
    mask = np.random.random(len(df)) < 0.10
    df.loc[mask, col] = np.nan

print('пропуски:', df.isna().sum().sum())

пропуски: 130


## Разбиение train/val/test

In [74]:
X = df[feature_names].values.astype(float)
y = df[target_col].values.astype(int)
classes = np.unique(y[~np.isnan(y)]).astype(int)

X_tv, X_test, y_tv, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_tv, y_tv, test_size=0.25, random_state=42, stratify=y_tv)

print(f'train={len(y_train)} val={len(y_val)} test={len(y_test)}')

train=181 val=61 test=61


## Построение дерева

In [75]:
tree = tree_growing(X_train, y_train, cat_features, classes)
print(f'глубина={tree_depth(tree)} листьев={count_leaves(tree)}')

глубина=8 листьев=30


## Качество до редукции

In [76]:
acc_train = accuracy(y_train, predict(tree, X_train, classes))
acc_test_before = accuracy(y_test, predict(tree, X_test, classes))
f1_before = f1_score(y_test, predict(tree, X_test, classes))

print(f'train acc={acc_train:.3f}')
print(f'test  acc={acc_test_before:.3f} f1={f1_before:.3f}')

train acc=0.978
test  acc=0.689 f1=0.740


## Редукция (pruning)

In [77]:
prune(tree, X_val, y_val, classes)
print(f'глубина={tree_depth(tree)} листьев={count_leaves(tree)}')

глубина=6 листьев=9


## Качество после редукции

In [66]:
acc_test_after = accuracy(y_test, predict(tree, X_test, classes))
f1_after = f1_score(y_test, predict(tree, X_test, classes))

print(f'test acc={acc_test_after:.3f} f1={f1_after:.3f}')

test acc=0.639 f1=0.703


## Сравнение со sklearn

In [67]:
df_filled = df.copy()
for col in num_columns:
    df_filled[col] = df_filled[col].fillna(df_filled[col].median())
for col in cat_columns:
    df_filled[col] = df_filled[col].fillna(df_filled[col].mode()[0])

X_f = df_filled[feature_names].values.astype(float)
y_f = df_filled[target_col].values.astype(int)
X_tr, X_te, y_tr, y_te = train_test_split(X_f, y_f, test_size=0.2, random_state=42, stratify=y_f)

clf = DecisionTreeClassifier(criterion='gini', random_state=42)
clf.fit(X_tr, y_tr)
y_sk = clf.predict(X_te)

acc_sk = accuracy_score(y_te, y_sk)
f1_sk = f1_score(y_te, y_sk)
print(f'sklearn acc={acc_sk:.3f} f1={f1_sk:.3f}')

sklearn acc=0.721 f1=0.761


## Итоговая таблица

In [68]:
pd.DataFrame({
    'модель': ['ID3 до редукции', 'ID3 после редукции', 'sklearn'],
    'accuracy': [acc_test_before, acc_test_after, acc_sk],
    'f1': [f1_before, f1_after, f1_sk],
})

,модель,accuracy,f1
0,ID3 до редукции,0.688525,0.739726
1,ID3 после редукции,0.639344,0.702703
2,sklearn,0.721311,0.760563
